In [1]:
from langchain_community.document_loaders import PyPDFLoader

pdf_files = [
    r"D:\\BE\\AI Adv proj\\data.pdf",
    r"D:\\BE\\AI Adv proj\\data2.pdf"
]

documents = []

for file in pdf_files:
    loader = PyPDFLoader(file)
    documents.extend(loader.load())

c:\Users\SPOORTHI\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)

chunks = text_splitter.split_documents(documents)

print(len(chunks))

3392


In [3]:
from langchain_community.embeddings import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

print("Embedding model loaded")

C:\Users\SPOORTHI\AppData\Local\Temp\ipykernel_29912\173910916.py:3: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(


Embedding model loaded


In [4]:
from langchain_community.vectorstores import FAISS

vector_db = FAISS.from_documents(chunks, embedding_model)

vector_db.save_local("faiss_index")

print("Vector DB created")

Vector DB created


In [5]:
retriever = vector_db.as_retriever(search_kwargs={"k": 5})
print("Retriever created")

Retriever created


In [ ]:
import os
from langchain_google_genai import ChatGoogleGenerativeAI

os.environ["GOOGLE_API_KEY"] = "Insert key here"

llm = ChatGoogleGenerativeAI(
    model="gemini-3-flash-preview",   # stable + fast
    temperature=0
)

In [7]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# Create prompt
prompt = ChatPromptTemplate.from_template("""
You are an intelligent AI tutor.

Use the provided context as your primary reference.
But you are also allowed to use your own knowledge to:
- explain concepts clearly
- elaborate with examples
- simplify for better understanding

If the context contains relevant information:
- use it first
- then expand on it

If the context is insufficient:
- use your own knowledge, but clearly explain
                                          
If possible, give examples and real-world applications.

Context:
{context}

Question:
{question}

Answer like a teacher:
""")
# Retriever
retriever = vector_db.as_retriever()

# Format documents
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# RAG pipeline
rag_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | prompt
    | llm
    | StrOutputParser()
)

# Test
response = rag_chain.invoke("Explain machine learning")
print(response)

Hello! I’m happy to help you understand Machine Learning. Think of me as your tutor, and we’ll break this down step-by-step using the information from your textbook, along with some helpful examples to make it stick.

### What is Machine Learning?

At its simplest level, **Machine Learning (ML)** is a field where we teach computers to learn from data so they can make decisions on their own. 

According to the context provided, the main goal of ML is to create models that are easy to understand, moving them beyond just "research projects" and into real-world tools that businesses can use every day.

---

### How Does It Work? (The "Learning" Process)

In traditional computer programming, a human writes specific rules for the computer to follow (like a recipe). In **Machine Learning**, the process is different:

1.  **Input Data + Expected Outputs:** We give the computer a lot of examples (input data) and tell it what the result should be (expected outputs).
2.  **Pattern Recognition:** 

In [8]:
response = rag_chain.invoke("what are loops in programming? Explain in detail like a beginner")
print(response)


Hello there! I am your AI tutor, and I’m excited to help you understand one of the most powerful tools in a programmer's toolkit: **Loops**.

Think of a loop as a way to tell a computer, "Hey, see this task? Keep doing it until I tell you to stop or until you've finished the list I gave you."

Let’s break this down step-by-step.

---

### 1. What is a Loop?
In programming, a loop is used for **iterating over a sequence**. As the provided text mentions, loops are likely the most useful feature in any programming language. Instead of writing the same line of code 10, 100, or 1,000 times, you write it once inside a loop, and the computer handles the repetition for you.

### 2. The "For Loop" (The Most Common Type)
The context introduces the **For Loop**. This is specifically used when you want to go through a sequence (like a list of numbers or names).

**Example from the text:**
```python
for i in range(1, 10):
    print(i)
```
**How to read this like a human:**
*   **`for i`**: "For eve